In [1]:
!apt-get update
!apt-get install -y texlive-latex-base texlive-latex-extra texlive-fonts-recommended

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,852 kB]
Get:14 http://

In [2]:
# 1. Upload files
from google.colab import files

print("Upload summaries file (.txt)")
uploaded = files.upload()
txt_file = list(uploaded.keys())[0]

print("Upload equations.tex")
uploaded2 = files.upload()
eq_file = list(uploaded2.keys())[0]

Upload summaries file (.txt)


Saving all_summaries (3).txt to all_summaries (3).txt
Upload equations.tex


Saving equations (1).tex to equations (1).tex


In [6]:
# 2. Read files
with open(txt_file, encoding="utf-8") as f:
    summaries = f.read()

with open(eq_file, encoding="utf-8") as f:
    equations = f.read()

# 3. Fix broken LaTeX equations
import re

def fix_broken_lines(text):

    lines = text.splitlines()
    fixed = []
    buffer = ""

    for line in lines:

        line = line.strip()

        # remove spaces inside subscripts like {l o n g}
        line = re.sub(
            r'_{([^}]*)}',
            lambda m: "_{" + m.group(1).replace(" ", "") + "}",
            line
        )

        # fix excessive ~ spacing
        line = re.sub(r'~{3,}', r'\\qquad ', line)

        # fix common OCR mistakes
        line = line.replace("1009_{0}", "100\\%")
        line = line.replace("9_{0}", "\\%")

        buffer += " " + line

        # flush when equation block ends
        if line.endswith("]") or line.endswith("}") or line.endswith(")"):
            fixed.append(buffer.strip())
            buffer = ""

    if buffer:
        fixed.append(buffer.strip())

    return "\n".join(fixed)


equations = fix_broken_lines(equations)

# 4. Escape LaTeX characters
def escape_latex(text):

    replacements = {
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
    }

    for k, v in replacements.items():
        text = text.replace(k, v)

    return text

def shorten_title(title):

    if "Artificial Intelligence Technology" in title:
        return title.replace("Artificial Intelligence Technology", "AIT")

    return title

def convert_summary_to_latex(text):

    lines = text.splitlines()
    latex = []

    for line in lines:

        if line.startswith("FILE:"):
            title = line[5:].strip()

            title = title.replace("Artificial Intelligence Technology", "AIT")

            title = escape_latex(title)

            # same font as text, just bold
            latex.append(f"\\textbf{{{title}}}\\\\ ")

        elif line.startswith("TITLE:"):
            title = escape_latex(line[6:].strip())

            latex.append(f"\\textbf{{\\textcolor{{red}}{{{title}}}}}\\\\ ")

        elif line.startswith("-"):
            bullet = escape_latex(line[1:].strip())
            latex.append(f"$\\bullet$ {bullet}\\\\ ")

        else:
            latex.append(escape_latex(line) + "\\\\ ")

    return "\n".join(latex)


summary_latex = convert_summary_to_latex(summaries)

# 6. Build LaTeX document
latex_doc = f"""
\\documentclass[7pt]{{article}}

\\usepackage{{amsmath}}
\\usepackage{{amssymb}}
\\usepackage{{multicol}}
\\usepackage{{geometry}}
\\usepackage{{titlesec}}
\\usepackage{{xcolor}}   % <-- ADD THIS

\\geometry{{margin=0.4cm}}

\\setlength\\parindent{{0pt}}
\\setlength\\parskip{{0pt}}

\\titlespacing*{{\\section}}{{0pt}}{{2pt}}{{2pt}}

\\setlength{{\\abovedisplayskip}}{{3pt}}
\\setlength{{\\belowdisplayskip}}{{3pt}}

\\begin{{document}}

\\tiny

\\begin{{multicols}}{{4}}

{summary_latex}

\\columnbreak

\\scriptsize
{equations}

\\end{{multicols}}

\\end{{document}}
"""

# 7. Save TEX file
with open("cheat_sheet.tex", "w", encoding="utf-8") as f:
    f.write(latex_doc)


In [7]:
# 8. Compile PDF
!pdflatex -interaction=nonstopmode cheat_sheet.tex


This is pdfTeX, Version 3.141592653-2.6-1.40.22 (TeX Live 2022/dev/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./cheat_sheet.tex
LaTeX2e <2021-11-15> patch level 1
L3 programming layer <2022-01-21>
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2021/10/04 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsmath.sty
For additional information on amsmath, use the `?' option.
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amstext.sty
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsgen.sty))
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsbsy.sty)
(/usr/share/texlive/texmf-dist/tex/latex/amsmath/amsopn.sty))
(/usr/share/texlive/texmf-dist/tex/latex/amsfonts/amssymb.sty
(/usr/share/texlive/texmf-dist/tex/latex/amsfonts/amsfonts.sty))
(/usr/share/texlive/texmf-dist/tex/latex/tools/multicol.sty)
(/usr/share/te

In [8]:
# 9. Download PDF
import os
from google.colab import files

if os.path.exists("cheat_sheet.pdf"):
    print("PDF successfully created!")
    files.download("cheat_sheet.pdf")
else:
    print("PDF was not created. Check LaTeX errors above.")

PDF successfully created!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>